# AU alignment calibration

Calibrate `tau_mask`, `n_eff_min`, `q_c`, and `eta` using RAF-DB/ExpW train only. Do not use validation or test accuracy.

In [1]:
from pathlib import Path
import torch
from cs_aualign_vit.data.datasets import build_loader

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository or notebooks directory")

SOURCE_MANIFEST = PROJECT_ROOT / "data/processed/raf-v5-deduplicated/processed.jsonl"
TARGET_MANIFEST = Path("/home/hung/FER-DomainAdaptation/CS-AUAlign-ViT/data/processed/expw-v1/processed.jsonl")
SOURCE_CHECKPOINT = PROJECT_ROOT / "runs/source-pretraining-val/best.pt"
ADAPTATION_CHECKPOINT = PROJECT_ROOT / "runs/adaptation/run-001/adaptation.pt"
SOURCE_AU_CACHE = PROJECT_ROOT / "runs/au_cache/raf_db_train.csv"
TARGET_AU_CACHE = PROJECT_ROOT / "runs/au_cache/expw_train.csv"
BATCH_SIZE = 64
SEED = 42
N_EFF_CANDIDATES = (2, 4, 8, 16)
R_TARGET = 0.1

In [2]:
source_loader = build_loader(SOURCE_MANIFEST, "source_train", batch_size=BATCH_SIZE, seed=SEED, au_cache=SOURCE_AU_CACHE)
target_loader = build_loader(TARGET_MANIFEST, "target_adaptation", batch_size=BATCH_SIZE, seed=SEED, au_cache=TARGET_AU_CACHE)

def collect(loader):
    au, mass, au_valid, landmark_valid = [], [], [], []
    for _images, _identity, metadata in loader:
        au.append(metadata["au"])
        mass.append(metadata["mask"].sum(dim=1))
        au_valid.append(metadata["au_valid"])
        landmark_valid.append(metadata["landmark_valid"])
    return tuple(torch.cat(values) for values in (au, mass, au_valid, landmark_valid))

source = collect(source_loader)
target = collect(target_loader)
q_c = source[0].mean(dim=0) * target[0].mean(dim=0)
q_c

tensor([0.0005, 0.0002, 0.0094, 0.0004, 0.0004, 0.0007, 0.0018, 0.0003, 0.0007,
        0.0003, 0.0110, 0.0011])

In [3]:
def mask_report(values):
    au, mass, au_valid, landmark_valid = values
    quantiles = torch.tensor([0.01, 0.05, 0.10, 0.50, 0.95])
    return {
        "minimum": mass.amin(dim=0),
        "quantiles": torch.quantile(mass, quantiles, dim=0),
        "near_zero_rate": (mass <= 1e-8).float().mean(dim=0),
        "au_missing_rate": (~au_valid).float().mean(),
        "landmark_invalid_rate": (~landmark_valid).float().mean(),
    }

source_mask_report = mask_report(source)
target_mask_report = mask_report(target)
source_mask_report, target_mask_report

({'minimum': tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]),
  'quantiles': tensor([[2.7697, 2.7697, 2.7697, 2.8834, 2.8834, 1.6886, 3.4390, 3.4390, 2.1221,
           3.4390, 3.4390, 3.4390],
          [4.3553, 4.3553, 4.3553, 4.3933, 4.3933, 2.8944, 5.2769, 5.2769, 3.0605,
           5.2769, 5.2769, 5.2769],
          [5.0577, 5.0577, 5.0577, 5.0806, 5.0806, 3.4946, 6.1750, 6.1750, 3.4578,
           6.1750, 6.1750, 6.1750],
          [6.6237, 6.6237, 6.6237, 6.6318, 6.6318, 4.6663, 8.0508, 8.0508, 4.4095,
           8.0508, 8.0508, 8.0508],
          [7.5666, 7.5666, 7.5666, 7.5868, 7.5868, 5.4798, 9.2320, 9.2320, 5.2953,
           9.2320, 9.2320, 9.2320]]),
  'near_zero_rate': tensor([0.0024, 0.0024, 0.0024, 0.0024, 0.0024, 0.0024, 0.0024, 0.0024, 0.0024,
          0.0024, 0.0024, 0.0024]),
  'au_missing_rate': tensor(0.0163),
  'landmark_invalid_rate': tensor(0.0024)},
 {'minimum': tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]),
  'quantiles': tensor([[0.0000,

In [4]:
TAU_MASK = 1.0

def effective_sizes(metadata, tau_mask):
    mass = metadata["mask"].sum(dim=1)
    valid = (
        metadata["au_valid"][:, None]
        & metadata["landmark_valid"][:, None]
        & (mass >= tau_mask)
    )
    weights = metadata["au"] * valid
    sums = weights.sum(dim=0)
    n_eff = sums.square() / (weights.square().sum(dim=0) + 1e-8)
    return n_eff, sums

source_train_loader = build_loader(
    SOURCE_MANIFEST, "source_train",
    batch_size=BATCH_SIZE, seed=SEED, drop_last=True,
    au_cache=SOURCE_AU_CACHE,
)
target_train_loader = build_loader(
    TARGET_MANIFEST, "target_adaptation",
    batch_size=BATCH_SIZE, seed=SEED, drop_last=True,
    au_cache=TARGET_AU_CACHE,
)

source_rows, target_rows = [], []
source_iterator = iter(source_train_loader)

for _target_images, _target_ids, target_metadata in target_train_loader:
    try:
        _source_images, _source_labels, source_metadata = next(source_iterator)
    except StopIteration:
        source_iterator = iter(source_train_loader)
        _source_images, _source_labels, source_metadata = next(source_iterator)

    source_rows.append(effective_sizes(source_metadata, TAU_MASK))
    target_rows.append(effective_sizes(target_metadata, TAU_MASK))

def stack_rows(rows):
    n_eff, sums = zip(*rows)
    return torch.stack(n_eff), torch.stack(sums)

source_n_eff, source_sums = stack_rows(source_rows)
target_n_eff, target_sums = stack_rows(target_rows)

for n_min in N_EFF_CANDIDATES:
    source_valid = (source_n_eff >= n_min) & (source_sums > 1e-8)
    target_valid = (target_n_eff >= n_min) & (target_sums > 1e-8)
    paired_valid = source_valid & target_valid

    print(
        f"n_eff_min={n_min:2d} | "
        f"source AU hợp lệ={source_valid.float().mean():.1%} | "
        f"target AU hợp lệ={target_valid.float().mean():.1%} | "
        f"AU hợp lệ cả hai={paired_valid.float().mean():.1%} | "
        f"batch có local loss={paired_valid.any(dim=1).float().mean():.1%}"
    )

n_eff_min= 2 | source AU hợp lệ=100.0% | target AU hợp lệ=100.0% | AU hợp lệ cả hai=100.0% | batch có local loss=100.0%
n_eff_min= 4 | source AU hợp lệ=100.0% | target AU hợp lệ=99.7% | AU hợp lệ cả hai=99.7% | batch có local loss=100.0%
n_eff_min= 8 | source AU hợp lệ=97.2% | target AU hợp lệ=94.7% | AU hợp lệ cả hai=92.8% | batch có local loss=100.0%
n_eff_min=16 | source AU hợp lệ=69.6% | target AU hợp lệ=64.9% | AU hợp lệ cả hai=53.2% | batch có local loss=100.0%


## Gradient calibration

Using a fixed list of train batches and the same encoder parameters, measure `||grad L_local|| / ||grad L_POST||`. For each batch compute `eta_0 = R_TARGET * ||grad L_POST|| / (||grad L_local|| + epsilon)`. Choose one fixed `eta` from the median and report P10/P90. Fail calibration when local gradients are almost always zero. Copy the selected values and supporting measurements to `docs/experiments/au_alignment_calibration.md`, then update `configs/training/adaptation.yaml`.

In [5]:
import math
from itertools import islice

import yaml

from cs_aualign_vit.models import (
    POSTBCSModel,
    PrototypeBanks,
    AdversarialAlignment,
)
from cs_aualign_vit.training.au_alignment import compute_au_alignment
from cs_aualign_vit.training.prototype_learning import compute_prototype_losses
from cs_aualign_vit.training.target_supervision import (
    refresh_target_supervision,
)

N_EFF_MIN = 8
PILOT_BATCHES = 4

config = yaml.safe_load(
    (PROJECT_ROOT / "configs/training/adaptation.yaml").read_text()
)
device = torch.device(config["training"]["device"])
torch.manual_seed(SEED)

model = POSTBCSModel.from_source_checkpoint(SOURCE_CHECKPOINT).to(device)
banks = PrototypeBanks.from_source_checkpoint(
    SOURCE_CHECKPOINT, feature_dim=model.feature_dim
).to(device)
adversarial = AdversarialAlignment(
    input_dim=model.feature_dim, **config["adversarial"]
).to(device)

# Target-feature loader không dùng nhãn target hoặc AU cache.
target_feature_loader = build_loader(
    TARGET_MANIFEST,
    "target_features",
    batch_size=BATCH_SIZE,
    seed=SEED,
    **config["preprocessing"],
)
target_sample_ids = [
    sample["sample_id"]
    for sample in target_feature_loader.dataset.samples
]

# Quét ExpW train một lần để tạo pseudo labels epoch 0.
# Hàm này không cập nhật weights.
supervision = refresh_target_supervision(
    model,
    banks,
    target_feature_loader,
    target_sample_ids,
    epoch=0,
)
assert supervision.epoch == 0
print(
    "Pseudo labels epoch 0:",
    len(supervision.sample_ids),
    "samples,",
    int(supervision.trusted_mask.sum()),
    "trusted",
)

model.train()
banks.train()
adversarial.train()

# Cùng tập encoder parameters cho hai phép đo gradient.
encoder_parameters = tuple(
    parameter
    for parameter in model.bcs_block1.parameters()
    if parameter.requires_grad
)

def gradient_norm(loss, parameters, *, retain_graph):
    gradients = torch.autograd.grad(
        loss,
        parameters,
        retain_graph=retain_graph,
        allow_unused=True,
    )
    squares = [
        gradient.detach().float().square().sum()
        for gradient in gradients
        if gradient is not None
    ]
    return (
        torch.stack(squares).sum().sqrt()
        if squares
        else torch.tensor(0.0, device=loss.device)
    )

rows = []
for source_batch, target_batch in islice(
    zip(source_train_loader, target_train_loader), PILOT_BATCHES
):
    source_images, source_labels, source_au = source_batch
    target_images, sample_ids, target_au = target_batch
    pseudo_labels, trusted_mask = supervision.lookup(list(sample_ids))

    source_images = source_images.to(device)
    source_labels = source_labels.to(device)
    target_images = target_images.to(device)

    with torch.autocast(
        device_type=device.type,
        dtype=torch.float16,
        enabled=(device.type == "cuda" and config["training"]["amp"]),
    ):
        features = model.forward_features(source_images, target_images)
        losses = compute_prototype_losses(
            features,
            banks,
            source_labels,
            pseudo_labels,
            trusted_mask,
            num_classes=7,
        )
        loss_ad = adversarial(
            features.source_features,
            features.target_features,
        )
        loss_post = (
            losses.l_s
            + losses.l_t
            + config["loss"]["alpha"] * losses.l_t2s
            + config["loss"]["beta"] * losses.l_s2t
            + config["loss"]["gamma"] * loss_ad
        )
        alignment = compute_au_alignment(
            features.source_native_tokens,
            features.target_native_tokens,
            source_au,
            target_au,
            tau_mask=TAU_MASK,
            n_eff_min=N_EFF_MIN,
            q_c=q_c,
        )
        loss_local = alignment.loss

    post_norm = gradient_norm(
        loss_post, encoder_parameters, retain_graph=True
    )
    local_norm = gradient_norm(
        loss_local, encoder_parameters, retain_graph=False
    )

    row = {
        "loss_post": float(loss_post.detach()),
        "loss_local": float(loss_local.detach()),
        "valid_aus": int(alignment.valid_aus.sum()),
        "post_grad_norm": float(post_norm),
        "local_grad_norm": float(local_norm),
        "raw_ratio": float(local_norm / (post_norm + 1e-8)),
        "eta_0": float(R_TARGET * post_norm / (local_norm + 1e-8)),
    }
    rows.append(row)
    print(row)

    del features, losses, loss_ad, loss_post, alignment, loss_local
    del post_norm, local_norm

valid_eta = torch.tensor([
    row["eta_0"]
    for row in rows
    if row["local_grad_norm"] > 1e-8
    and math.isfinite(row["eta_0"])
])
if len(valid_eta) < max(2, len(rows) // 2):
    raise RuntimeError(
        "Local gradient gần như luôn bằng 0; kiểm tra AU coverage"
    )

ETA = float(torch.quantile(valid_eta, 0.5))
print({
    "calibration_epoch": 0,
    "tau_mask": TAU_MASK,
    "n_eff_min": N_EFF_MIN,
    "q_c": [float(value) for value in q_c],
    "eta": ETA,
    "eta_p10": float(torch.quantile(valid_eta, 0.10)),
    "eta_p90": float(torch.quantile(valid_eta, 0.90)),
    "zero_local_gradient_batches": sum(
        row["local_grad_norm"] <= 1e-8 for row in rows
    ),
})

Pseudo labels epoch 0: 65948 samples, 25236 trusted
{'loss_post': 1.4948691129684448, 'loss_local': 151.07122802734375, 'valid_aus': 11, 'post_grad_norm': 3.7213923931121826, 'local_grad_norm': 583.174072265625, 'raw_ratio': 156.7085723876953, 'eta_0': 0.0006381272105500102}
{'loss_post': 1.0905200242996216, 'loss_local': 105.1829605102539, 'valid_aus': 11, 'post_grad_norm': 3.1247315406799316, 'local_grad_norm': 395.1603698730469, 'raw_ratio': 126.4621810913086, 'eta_0': 0.000790750200394541}
{'loss_post': 0.9057273864746094, 'loss_local': 90.55179595947266, 'valid_aus': 12, 'post_grad_norm': 3.2552742958068848, 'local_grad_norm': 347.7025451660156, 'raw_ratio': 106.81205749511719, 'eta_0': 0.0009362238924950361}
{'loss_post': 1.909516453742981, 'loss_local': 103.39351654052734, 'valid_aus': 10, 'post_grad_norm': 4.929556846618652, 'local_grad_norm': 414.95245361328125, 'raw_ratio': 84.17642211914062, 'eta_0': 0.0011879811063408852}
{'calibration_epoch': 0, 'tau_mask': 1.0, 'n_eff_min